In [0]:
%sql
-- Silver involves several decisions: what type each column gets, how to handle hidden values, which columns become dimension tables. Rather than guess, we’ll query bronze first and let the results drive those decisions.

SELECT 'GE65_Sprsn_Flag' AS flag_column, GE65_Sprsn_Flag AS value, COUNT(*) AS n
FROM cms.bronze.partd_prescriber_drug
GROUP BY GE65_Sprsn_Flag

UNION ALL

SELECT 'GE65_Bene_Sprsn_Flag', GE65_Bene_Sprsn_Flag, COUNT(*)
FROM cms.bronze.partd_prescriber_drug
GROUP BY GE65_Bene_Sprsn_Flag

ORDER BY flag_column, n DESC;

-- For this, we're asking what values the flags actually take. Most are just * or #. Some are missing. 
-- The 65-and-older patient counts are hidden on 88% of rows, and the 65+ claim totals on 44%. The 65+ columns are of limited use, so the analysis should be built on the overall totals.

In [0]:
%sql
SELECT
  SUM(CASE WHEN Tot_Clms          IS NOT NULL AND try_cast(Tot_Clms          AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_tot_clms,
  SUM(CASE WHEN Tot_30day_Fills   IS NOT NULL AND try_cast(Tot_30day_Fills   AS DECIMAL(18,1)) IS NULL THEN 1 ELSE 0 END) AS bad_tot_30day,
  SUM(CASE WHEN Tot_Day_Suply     IS NOT NULL AND try_cast(Tot_Day_Suply     AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_day_suply,
  SUM(CASE WHEN Tot_Drug_Cst      IS NOT NULL AND try_cast(Tot_Drug_Cst      AS DECIMAL(18,2)) IS NULL THEN 1 ELSE 0 END) AS bad_drug_cst,
  SUM(CASE WHEN Tot_Benes         IS NOT NULL AND try_cast(Tot_Benes         AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_tot_benes,
  SUM(CASE WHEN GE65_Tot_Drug_Cst IS NOT NULL AND try_cast(GE65_Tot_Drug_Cst AS DECIMAL(18,2)) IS NULL THEN 1 ELSE 0 END) AS bad_ge65_cst
FROM cms.bronze.partd_prescriber_drug;

-- Now we ask will every numeric column convert cleanly? 

In [0]:
## Cell 2b

from pyspark.sql import functions as F

numeric_types = {
    "Tot_Clms":             "BIGINT",
    "Tot_30day_Fills":      "DECIMAL(18,1)",
    "Tot_Day_Suply":        "BIGINT",
    "Tot_Drug_Cst":         "DECIMAL(18,2)",
    "Tot_Benes":            "BIGINT",
    "GE65_Tot_Clms":        "BIGINT",
    "GE65_Tot_30day_Fills": "DECIMAL(18,1)",
    "GE65_Tot_Day_Suply":   "BIGINT",
    "GE65_Tot_Drug_Cst":    "DECIMAL(18,2)",
    "GE65_Tot_Benes":       "BIGINT",
}

bronze = spark.table("cms.bronze.partd_prescriber_drug")

checks = []
for c, t in numeric_types.items():
    # values that are present but fail conversion
    checks.append(
        F.sum(F.when(F.col(c).isNotNull() & F.expr(f"try_cast({c} AS {t})").isNull(), 1)
               .otherwise(0)).alias(f"bad_{c}")
    )
    # most digits after the decimal point anywhere in the column
    checks.append(
        F.max(F.length(F.regexp_extract(F.col(c), r"\.(\d+)$", 1))).alias(f"maxdp_{c}")
    )

row = bronze.agg(*checks).first().asDict()

for c, t in numeric_types.items():
    print(f"{c:22} {t:14} bad={row[f'bad_{c}']:<3} max_decimals={row[f'maxdp_{c}']}")

## We added a new Python cell that now can deal with rounding by taking into account max decimals, which can be widened if any columns are added in the future requiring it. This supplants Cell 2, but we're keeping it for legacy purposes. 

In [0]:
%sql
SELECT
  COUNT(*)                                            AS total_rows,
  SUM(CASE WHEN Tot_Benes IS NULL THEN 1 ELSE 0 END)  AS tot_benes_hidden,
  ROUND(100 * AVG(CASE WHEN Tot_Benes IS NULL THEN 1 ELSE 0 END), 1) AS pct_benes_hidden,
  SUM(CASE WHEN Tot_Drug_Cst IS NULL THEN 1 ELSE 0 END) AS drug_cst_missing
FROM cms.bronze.partd_prescriber_drug;

-- How much is hidden overall.
-- Tot_Benes is hidden on 54.5% of rows. That means the reliable metrics are claims and cost, like cost per claim.

In [0]:
%sql
SELECT
  COUNT(DISTINCT Prscrbr_NPI)   AS distinct_prescribers,
  COUNT(DISTINCT Brnd_Name)     AS distinct_brands,
  COUNT(DISTINCT Gnrc_Name)     AS distinct_generics,
  COUNT(DISTINCT Prscrbr_Type)  AS distinct_specialties,
  COUNT(DISTINCT Prscrbr_State_Abrvtn) AS distinct_states,
  MIN(LENGTH(Prscrbr_NPI))      AS min_npi_len,
  MAX(LENGTH(Prscrbr_NPI))      AS max_npi_len
FROM cms.bronze.partd_prescriber_drug;

-- How many distinct values does each dimension have?

In [0]:
%sql
SELECT Prscrbr_Type, COUNT(*) AS n_rows, COUNT(DISTINCT Prscrbr_NPI) AS n_prescribers
FROM cms.bronze.partd_prescriber_drug
WHERE lower(Prscrbr_Type) LIKE '%dent%'
   OR lower(Prscrbr_Type) LIKE '%oral%'
GROUP BY Prscrbr_Type
ORDER BY n_rows DESC;

-- Dental prescrbers

In [0]:
%sql
SELECT
  GE65_Sprsn_Flag,
  GE65_Bene_Sprsn_Flag,
  COUNT(*)                                                      AS n,
  SUM(CASE WHEN GE65_Tot_Clms      IS NULL THEN 1 ELSE 0 END)   AS clms_null,
  SUM(CASE WHEN GE65_Tot_Drug_Cst  IS NULL THEN 1 ELSE 0 END)   AS cst_null,
  SUM(CASE WHEN GE65_Tot_Day_Suply IS NULL THEN 1 ELSE 0 END)   AS suply_null,
  SUM(CASE WHEN GE65_Tot_Benes     IS NULL THEN 1 ELSE 0 END)   AS benes_null
FROM cms.bronze.partd_prescriber_drug
GROUP BY GE65_Sprsn_Flag, GE65_Bene_Sprsn_Flag
ORDER BY GE65_Sprsn_Flag, GE65_Bene_Sprsn_Flag;

-- Which flag actually controls the 65+ cost and days’ supply? GE65_Sprsn_Flag controls cost, then cst_null will equal n on every row where that flag is * or #, and be 0 where it’s null, no matter what the other flag says.
-- The CMS dictionary says * means the value itself is between 1 and 10, and # means the value was hidden because the unseen under-65 portion is between 1 and 10, so showing the 65+ number would let anyone work out the hidden one by subtraction.

In [0]:
%sql
WITH typed AS (
  SELECT
    try_cast(Tot_Clms        AS BIGINT)        AS clms,
    try_cast(Tot_30day_Fills AS DECIMAL(18,1)) AS fills
  FROM cms.bronze.partd_prescriber_drug
)
SELECT
  SUM(CASE WHEN fills < clms      THEN 1 ELSE 0 END) AS fills_below_claims,
  SUM(CASE WHEN fills > 12 * clms THEN 1 ELSE 0 END) AS fills_above_12x,
  MIN(clms)                                          AS min_claims,
  ROUND(MIN(fills / clms), 2)                        AS min_ratio,
  ROUND(MAX(fills / clms), 2)                        AS max_ratio
FROM typed;

-- Testing the fills rule. Literally: claims ≤ fills ≤ 12 × claims. It converts the two columns once, under short names, so the main query isn’t repeating try_cast six times.